# Genie Agent — Sample Demand Queries

This notebook demonstrates the **Genie Conversation API** to ask business
questions about demand forecasts programmatically. Each cell submits a
natural-language question, waits for SQL generation and execution, and
displays the result.

**Prerequisites:** Run `setup_genie_agent` first to create the Genie space.

In [0]:
import time
from databricks.sdk import WorkspaceClient

GENIE_SPACE_ID = "01f1c22c3f621207842ec9d6577c22c7"
w = WorkspaceClient()

def ask_genie(question: str, space_id: str = GENIE_SPACE_ID) -> dict:
    """Submit a question to the Genie Agent and wait for the answer."""
    conv = w.genie.start_conversation(space_id=space_id, content=question)
    for _ in range(60):
        msg = w.genie.get_message(
            space_id=space_id,
            conversation_id=conv.conversation_id,
            message_id=conv.message_id,
        )
        if msg.status in ("COMPLETED", "FAILED"):
            break
        time.sleep(2)
    result = {"status": msg.status, "question": question}
    if hasattr(msg, 'attachments') and msg.attachments:
        for att in msg.attachments:
            if hasattr(att, 'query'):
                result["sql"] = att.query.query
                if hasattr(att.query, 'result'):
                    result["columns"] = att.query.result.columns
                    result["data"] = att.query.result.data_array
            if hasattr(att, 'text'):
                result["narrative"] = att.text.content
    return result

def display_genie(result: dict):
    print(f"Q: {result['question']}")
    print(f"Status: {result['status']}")
    if 'sql' in result:
        print(f"SQL: {result['sql'][:200]}...")
    if 'narrative' in result:
        print(f"Answer: {result['narrative']}")
    if 'data' in result and result['data']:
        import pandas as pd
        col_names = [c.name for c in result['columns']] if result.get('columns') else None
        df = pd.DataFrame(result['data'], columns=col_names)
        display(df.head(10))

In [0]:
r = ask_genie("Which model has the lowest average sMAPE across all time series?")
display_genie(r)

In [0]:
r = ask_genie("Compare the top 5 models for time series M1 by average metric value")
display_genie(r)

In [0]:
r = ask_genie("How many unique time series are in the training data, and what is the date range?")
display_genie(r)

In [0]:
r = ask_genie("What percentage of models have non-empty prediction intervals in the scoring output?")
display_genie(r)